# 06 — QAT Inference (pytorch-quantization INT8, 4-bit input)

Loads the best QAT checkpoint (`checkpoints/resnet18_qat_int8_in4b_cuda_bs256/qat_phase1_best.pth`),
runs evaluation on the ImageNet-100 validation set, and saves a `result.json` in the same format
as the PTQ notebooks so it can be loaded by `load_runs` / `flatten_runs`.

## 1. Imports & path setup

In [1]:
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT = Path("..").resolve()          # quantized_resnets/
SRC  = ROOT / "src"
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(SRC / "qat"))

from config import ExperimentConfig
from data import build_imagenet_dataset
from metrics import MetricsTracker
from qat.quantize import (
    get_quantized_model,
    initialize_quant_modules,
    setup_quantization_descriptors,
)

from pytorch_quantization import nn as quant_nn

## 2. Config — edit values here

In [2]:
DATA_DIR         = "/home/pf4636/imagenet"
CHECKPOINT       = str(ROOT / "checkpoints" / "resnet18_qat_int8_in4b_cuda_bs256" / "qat_phase1_best.pth")

INPUT_QUANT_BITS = 4     # must match the checkpoint's training config
NUM_CLASSES      = 100
BATCH_SIZE       = 256
NUM_WORKERS      = 8
NUM_EVAL_BATCHES = 500   # None = full val set; set to an int to limit
SEED             = 42

# Derived run identifiers (mirrors the PTQ run_id convention)
BACKEND          = "qat_pytorch"
MODEL_PRECISION  = "int8"
DEVICE_STR       = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ID           = f"resnet18_{BACKEND}_{MODEL_PRECISION}_in{INPUT_QUANT_BITS}b_{DEVICE_STR}_bs{BATCH_SIZE}"
RUNS_DIR         = ROOT / "runs"

print(f"run_id   : {RUN_ID}")
print(f"checkpoint: {CHECKPOINT}")

run_id   : resnet18_qat_pytorch_int8_in4b_cuda_bs256
checkpoint: /home/pf4636/code/resnet/quantized_resnets/checkpoints/resnet18_qat_int8_in4b_cuda_bs256/qat_phase1_best.pth


## 3. Reproducibility & device

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device(DEVICE_STR)
print(f"[Device] {device}")

[Device] cuda


## 4. Quantization setup & model load

`setup_quantization_descriptors` + `initialize_quant_modules` must run **before** the model
is instantiated so that `nn.Conv2d` / `nn.Linear` are already patched to their quantized
equivalents when `get_quantized_model` builds ResNet-18.

After loading the checkpoint state dict we explicitly enable fake-quant on every
`TensorQuantizer` (the amax values were saved in the checkpoint, so no re-calibration
is needed).

In [5]:
setup_quantization_descriptors()
initialize_quant_modules()

# Loads FP32 checkpoint path — but here we pass the QAT checkpoint directly.
# get_quantized_model accepts any checkpoint whose state dict matches the
# quantized ResNet-18 architecture (QAT checkpoints qualify).
ckpt   = torch.load(CHECKPOINT, map_location="cpu")
state  = ckpt["model"] if "model" in ckpt else ckpt

from model import ResNet18
model = ResNet18(num_classes=NUM_CLASSES, pretrained=False)
model.load_state_dict(state)

# Activate fake-quant on all TensorQuantizers.
# The calibrated amax values are already in the state dict; we just need to
# make sure the quantizers are in fake-quant mode (not calibration mode).
for module in model.modules():
    if isinstance(module, quant_nn.TensorQuantizer):
        if module._calibrator is not None:
            module.enable_quant()
            module.disable_calib()
        else:
            module.enable()

model = model.to(device)
model.eval()
print(f"[Model] QAT checkpoint loaded from:\n  {CHECKPOINT}")


W0409 22:04:45.779582 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.782877 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.783329 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.783906 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.784305 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.784724 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.785064 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.785295 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.785620 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.786040 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.786307 139502095151424 tensor_quantizer.py:174] Disable MaxCalibrator
W0409 22:04:45.786576 139502095151424 tensor_quantizer.py:174] Di

[Model] QAT checkpoint loaded from:
  /home/pf4636/code/resnet/quantized_resnets/checkpoints/resnet18_qat_int8_in4b_cuda_bs256/qat_phase1_best.pth


## 5. Validation dataloader

In [6]:
data_cfg = ExperimentConfig(
    imagenet_path    = DATA_DIR,
    device           = DEVICE_STR,
    batch_size       = BATCH_SIZE,
    num_workers      = NUM_WORKERS,
    num_classes      = NUM_CLASSES,
    input_quant_bits = INPUT_QUANT_BITS,
    seed             = SEED,
)

val_ds = build_imagenet_dataset(data_cfg, "val")
val_loader = DataLoader(
    val_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
    drop_last   = True,
)

print(f"[Data] Val samples: {len(val_ds):,}  |  batches: {len(val_loader)}")

[data] Filtered to 5000 samples across 100 classes.
[Data] Val samples: 5,000  |  batches: 19


## 6. Inference

In [7]:
criterion     = nn.CrossEntropyLoss()
metrics       = MetricsTracker()
WARMUP_BATCHES = 30

total_batches     = len(val_loader)
effective_batches = total_batches if NUM_EVAL_BATCHES is None else min(total_batches, NUM_EVAL_BATCHES)
print(f"Evaluating on {effective_batches} batches (first {WARMUP_BATCHES} are warmup) ...")

t_eval_start = time.perf_counter()

with torch.inference_mode():
    for batch_idx, (images, targets) in enumerate(val_loader):
        if NUM_EVAL_BATCHES is not None and batch_idx >= NUM_EVAL_BATCHES:
            break

        if batch_idx == WARMUP_BATCHES:
            print(f"  --- Warmup complete ({WARMUP_BATCHES} batches) — metric collection started ---")

        images  = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if device.type == "cuda":
            torch.cuda.synchronize()

        infer_start = time.perf_counter()
        outputs = model(images)

        if device.type == "cuda":
            torch.cuda.synchronize()

        infer_time = time.perf_counter() - infer_start
        batch_time = infer_time  # no separate data-transfer timing here

        loss_value = float(criterion(outputs, targets).item())

        if batch_idx >= WARMUP_BATCHES:
            metrics.update(
                outputs      = outputs,
                targets      = targets,
                loss_value   = loss_value,
                batch_time_s = batch_time,
                infer_time_s = infer_time,
                batch_size   = int(images.shape[0]),
            )

        if batch_idx >= WARMUP_BATCHES and (batch_idx + 1) % 10 == 0:
            s = metrics.summary()
            print(
                f"  Batch [{batch_idx + 1}/{effective_batches}]  "
                f"Top-1: {s['top1_acc']:.2f}%  |  Top-5: {s['top5_acc']:.2f}%  |  "
                f"Infer: {s['infer_ms_avg']:.2f} ms/batch"
            )

total_eval_time = time.perf_counter() - t_eval_start
summary = metrics.summary()
print(f"\n[Done]  Top-1: {summary['top1_acc']:.4f}%  |  Top-5: {summary['top5_acc']:.4f}%  |  "
      f"Samples: {summary['total_samples']:,}  |  Wall time: {total_eval_time:.1f}s")

Evaluating on 19 batches (first 30 are warmup) ...

[Done]  Top-1: 0.0000%  |  Top-5: 0.0000%  |  Samples: 0  |  Wall time: 7.9s


## 7. Save result.json

In [ ]:
payload = {
    "status" : "ok",
    "run_id" : RUN_ID,
    "system" : {
        "timestamp"      : time.strftime("%Y-%m-%d %H:%M:%S"),
        "torch"          : torch.__version__,
        "cuda_available" : torch.cuda.is_available(),
        "device"         : DEVICE_STR,
    },
    "config" : {
        "imagenet_path"       : DATA_DIR,
        "device"              : DEVICE_STR,
        "batch_size"          : BATCH_SIZE,
        "num_workers"         : NUM_WORKERS,
        "seed"                : SEED,
        "num_classes"         : NUM_CLASSES,
        "input_quant_bits"    : INPUT_QUANT_BITS,
        "model_precision"     : MODEL_PRECISION,
        "backend"             : BACKEND,
        "num_eval_batches"    : NUM_EVAL_BATCHES,
        "checkpoint"          : CHECKPOINT,
    },
    "results": summary,
    "error"  : None,
    "total_eval_time_sec": round(total_eval_time, 3),
}

out_dir  = RUNS_DIR / RUN_ID
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "result.json"

with open(out_path, "w") as f:
    json.dump(payload, f, indent=2, sort_keys=True)

print(f"[Saved] {out_path}")
print(f"  top1_acc : {summary['top1_acc']:.4f}%")
print(f"  top5_acc : {summary['top5_acc']:.4f}%")
print(f"  infer_ms : {summary['infer_ms_avg']:.4f} ms")

## 8. Quick sanity check — load back via load_runs

In [ ]:
import pandas as pd
from utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)

runs = load_runs("../runs", status="ok")
rows = flatten_runs(runs)
df   = pd.DataFrame(rows)

df_qat = df[df["cfg.backend"] == "qat"].copy()
df_qat[[
    "run_id",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]]